# Ultra Marathon Data Analysis

## Project Overview
This project analyses ultra-marathon race results using Python, Pandas and data visualisation libraries to identify participation trends and athlete performance. It demonstrates data cleaning, transformation, exploratory data analysis and visualisation using a large real-world dataset.

### Skills Demonstrated
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Data Cleaning
- Exploratory Data Analysis
- Data Visualisation


## Data loading and imports


In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt

# Path to the extracted folder
extracted_folder = '/content/extracted/'

# Full path to your CSV
csv_file = os.path.join(extracted_folder, 'TWO_CENTURIES_OF_UM_RACES.csv')

# Load CSV into a DataFrame
df = pd.read_csv(csv_file)


## Column standardisation and duplicate removal


In [ ]:
df = df.rename(columns={'Year of event': 'year',
                   'Event dates': 'event_dates',
                   'Event name': 'name',
                   'Event distance/length': 'distance',
                   'Event number of finishers': 'no_finishers',
                   'Athlete performance': 'athlete_performance',
                   'Athlete club': 'athlete_club',
                   'Athlete country': 'athlete_country',
                   'Athlete year of birth': 'athlete_year_of_birth',
                   'Athlete gender': 'athlete_gender',
                   'Athlete age category': 'athlete_age_category',
                   'Athlete average speed': 'athlete_average_speed',
                   'Athlete ID': 'athlete_id'})

In [ ]:
df = df.drop('athlete_id', axis=1)


In [ ]:
df = df.loc[~df.duplicated()]
df = df.reset_index(drop=True)


## Age feature engineering


In [ ]:
df['athlete_age'] = df['athlete_age_category'].str.replace('[MWF]', '', regex=True)

In [ ]:
df['age_under'] = np.where(
    df['athlete_age'].str.startswith('U', na=False),
    'Y',
    'N'
)


In [ ]:
df = df.drop('athlete_age_category', axis=1)

In [ ]:
df['athlete_age'] = df['athlete_age'].str.replace('U', '', regex=False)

In [ ]:
df['athlete_age'] = df['athlete_age'].astype('Int64')

## Missing values and distance filtering


In [ ]:
df = df.dropna(subset=["athlete_gender"])


In [ ]:
df = df[df["distance"].str.contains('km', case=False, na=False)]


In [ ]:
df["distance"] = df["distance"].str.replace(" ", "", regex=False)


In [ ]:
df["distance"] = df["distance"].str.lower()


In [ ]:
df["distance_km"] = df["distance"].str.extract(r"(\d+\.?\d*)km")

In [ ]:
df["race_stages"] = df["distance"].str.extract(r"(\d+\.?\d*)Etappen")

In [ ]:
df["distance_km"] = df["distance_km"].astype(float)

In [ ]:
df["race_stages"] = df["race_stages"].astype(float)

## Time parsing and speed calculation


In [ ]:
mask = df["athlete_average_speed"].str.contains(":", na=False)


In [ ]:
df.loc[mask, "time"] = pd.to_timedelta(df.loc[mask, "athlete_average_speed"])

In [ ]:
df["hours_num"] = df["time"].dt.total_seconds() / 3600

In [ ]:
df["time"] = pd.to_timedelta(df["athlete_performance"])


In [ ]:
df["hours_num"] = df["time"].dt.total_seconds() / 3600

In [ ]:
df["speed"] = df["distance_km"] / df["hours_num"]

In [ ]:
df["speed"] = df["speed"].fillna(285/df["hours_num"])

## Race stage parsing and completed-race subset


In [ ]:
df["race_stages"] = df["distance"].str.extract(r"(\d+\.?\d*)etappen")

In [ ]:
df["race_stages"] = df["distance"].str.extract(r"(\d+\.?\d*)stages")

In [ ]:
df["race_stages"] = df["distance"].str.extract(r"(\d+\.?\d*)x")

In [ ]:
df["race_stages"] = df["distance"].str.extract(r"470km/(\d+\.?\d*)")

In [ ]:
onerace = df[df["race_stages"].isna()]


In [ ]:
onerace = onerace.drop(columns=["time", "race_stages", "athlete_average_speed"])


In [ ]:
completed = onerace[np.isfinite(onerace["speed"])]


## Final filtering and summary


In [ ]:
completed.groupby("athlete_gender")["speed"].describe()


In [ ]:
completed = completed[(completed["name"]!="Bílovecká 50 (CZE)")]


In [ ]:
completed = completed[(completed["name"]!="Pass Mountain 50 km Race (USA)")]


In [ ]:
completed = completed[(completed["speed"]<=20)]


In [ ]:
completed.groupby("athlete_gender")["speed"].describe()

,count,mean,std,min,25%,50%,75%,max
athlete_gender,,,,,,,,
F,1103337.0,6.990455,1.966094,0.192135,5.556032,6.941231,8.303732,17.673049
M,5026612.0,7.516395,2.287218,0.501892,5.832415,7.407936,8.946722,19.713425
X,22.0,7.416663,1.610854,4.912418,6.054625,7.583528,8.565377,10.757829


## Visualisation


In [ ]:
sample = completed.copy()

plt.scatter(
    sample.loc[sample["athlete_gender"]=="M", "distance_km"],
    sample.loc[sample["athlete_gender"]=="M", "speed"],
    s=1,
    alpha=1,
    label="Men"
)

plt.scatter(
    sample.loc[sample["athlete_gender"]=="F", "distance_km"],
    sample.loc[sample["athlete_gender"]=="F", "speed"],
    s=1,
    alpha=0.2,
    label="Women"
)

plt.legend()
plt.xlabel('Distance (km)')
plt.ylabel('Speed (km/h)')
plt.show()
